In [16]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import (
    densenet121, DenseNet121_Weights,
    resnet50, ResNet50_Weights,
    efficientnet_v2_m, EfficientNet_V2_M_Weights,
    alexnet, AlexNet_Weights,
)
from torchvision import transforms
from huggingface_hub import hf_hub_download


In [18]:
# Configuración de semillas para reproducibilidad
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")


Dispositivo utilizado: cuda


# EDA de las galaxias DESI/HSC con `enyasantos/galaxy-classification-v02`

En lugar de imágenes sintéticas, este notebook utiliza las primeras `N_MUESTRAS` imágenes del archivo HDF5 indicado. Las imágenes se redimensionan a 224×224 y se pasan por el ensemble publicado. Para cada una de las cuatro CNN se extrae el vector de 128 características previo a la capa final, obteniendo 512 características por galaxia.

Además, se conservan las predicciones de las 10 clases del modelo. El ensemble utiliza **mean voting**, por lo que se promedian las probabilidades de las cuatro CNN. Esto permite explorar la distribución morfológica estimada de la muestra y la confianza de las predicciones.

**Importante:** estas etiquetas corresponden a las clases de Galaxy10 SDSS, que es el conjunto con el que fue entrenado el modelo. Por tanto, en las imágenes DESI/HSC de este notebook deben interpretarse como **clasificaciones estimadas por el modelo**, no como etiquetas de verdad terreno.


In [ ]:
# ==========================================
# 1. CARGA DEL DATASET HDF5
# ==========================================
import numpy as np

PATH = "/content/drive/MyDrive/galaxiesML/DESI_HSC_mini2.npz"

data = np.load(PATH)
images_3chan = data["images"]

print(images_3chan.shape)


In [ ]:
# ==========================================
# 2. PREPARAR LAS IMÁGENES PARA EL ENSEMBLE
# ==========================================
# El modelo publicado recibe imágenes RGB.
# Aceptamos tanto NHWC como NCHW y, si hubiera más de 3 canales,
# usamos los primeros tres de forma explícita.

images = np.asarray(images_3chan)

if images.ndim != 4:
    raise ValueError(f'Se esperaba un arreglo 4-D de imágenes, pero se obtuvo {images.shape}')

# Convertir a NHWC.
if images.shape[-1] in (3, 4):
    images_rgb = images[..., :3]
elif images.shape[1] >= 3:
    images_rgb = np.transpose(images[:, :3, :, :], (0, 2, 3, 1))
else:
    raise ValueError(f'No se encontraron al menos 3 canales RGB en {images.shape}')

# Normalización a uint8 solamente si los datos no están ya en [0, 255].
# La transformación posterior de torchvision convierte la imagen a [0,1].
if np.issubdtype(images_rgb.dtype, np.floating):
    lo, hi = np.nanmin(images_rgb), np.nanmax(images_rgb)
    print(f'Rango original de las imágenes: [{lo:.4g}, {hi:.4g}]')
    if lo >= 0 and hi <= 1:
        images_rgb = (images_rgb * 255).clip(0, 255).astype(np.uint8)
    else:
        # Escalamiento global robusto para poder visualizar/procesar imágenes científicas.
        p1, p99 = np.nanpercentile(images_rgb, [1, 99])
        images_rgb = ((images_rgb - p1) / (p99 - p1 + 1e-8) * 255).clip(0, 255).astype(np.uint8)
else:
    images_rgb = images_rgb.clip(0, 255).astype(np.uint8)

print(f'Dimensiones finales para el modelo (NHWC): {images_rgb.shape}')


In [ ]:
# ==========================================\n
# 3. EXTRACCIÓN DE DESCRIPTORES Y PREDICCIONES (HOOKS)
# ==========================================\n
BATCH_SIZE = 32

feature_blocks = []
model_probabilities = {}

galaxy_models = load_galaxy_models()

with torch.inference_mode():
    for name, model in galaxy_models.items():
        model_features = []
        model_probs = []
        
        # 1. Variable auxiliar para almacenar la salida intermedia
        captured_layer_output = {}
        
        def get_activation():
            def hook(module, input, output):
                captured_layer_output['features'] = output
            return hook

        # 2. Registrar el hook en la capa Linear(256, 128) -> Índice 3 del clasificador
        if name == "ResNet50":
            hook_handle = model.fc[3].register_forward_hook(get_activation())
        else:
            hook_handle = model.classifier[3].register_forward_hook(get_activation())

        # 3. Inferencia por lotes
        for start in range(0, len(images_rgb), BATCH_SIZE):
            batch_np = images_rgb[start:start + BATCH_SIZE]
            batch = torch.stack([image_transform(img) for img in batch_np]).to(device)

            # En una Sola Pasada se calcula el Forward completo
            log_probs = model(batch)
            probs = torch.exp(log_probs)
            model_probs.append(probs.cpu().numpy())

            # El hook captura automáticamente las 128 características durante model(batch)
            feats_128 = captured_layer_output['features']
            model_features.append(feats_128.cpu().numpy())

        # 4. Remover el hook para liberar memoria/referencias
        hook_handle.remove()

        model_features = np.concatenate(model_features, axis=0)
        model_probs = np.concatenate(model_probs, axis=0)

        print(f'{name}: features={model_features.shape}, probabilities={model_probs.shape}')
        feature_blocks.append(model_features)
        model_probabilities[name] = model_probs

# Concatenar los 4 modelos (4 x 128 = 512 características por imagen)
features = np.concatenate(feature_blocks, axis=1)

# Mean voting para el ensemble
ensemble_probabilities = np.mean(
    np.stack(list(model_probabilities.values()), axis=0),
    axis=0
)
ensemble_predictions = np.argmax(ensemble_probabilities, axis=1)
ensemble_confidence = np.max(ensemble_probabilities, axis=1)

print(f'\nDimensiones de los descriptores concatenados: {features.shape}')
print(f'Dimensiones de las probabilidades del ensemble: {ensemble_probabilities.shape}')

In [ ]:
# ==========================================
# 4. DISTRIBUCIÓN DE LAS CLASIFICACIONES
# ==========================================
# El modelo fue entrenado para 10 clases de Galaxy10 SDSS.
# Las etiquetas son las definidas por el conjunto Galaxy10.
GALAXY10_CLASSES = {
    0: 'Disturbed Galaxies',
    1: 'Merging Galaxies',
    2: 'Round Smooth Galaxies',
    3: 'In-between Round Smooth Galaxies',
    4: 'Cigar Shaped Smooth Galaxies',
    5: 'Barred Spiral Galaxies',
    6: 'Unbarred Tight Spiral Galaxies',
    7: 'Unbarred Loose Spiral Galaxies',
    8: 'Edge-on Galaxies without Bulge',
    9: 'Edge-on Galaxies with Bulge',
}

# Predicción final del ensemble (mean voting).
predicted_class_names = [GALAXY10_CLASSES[i] for i in ensemble_predictions]

classification_counts = pd.Series(predicted_class_names).value_counts()
classification_distribution = pd.DataFrame({
    'Clase': list(GALAXY10_CLASSES.values()),
    'Código': list(GALAXY10_CLASSES.keys()),
})
classification_distribution['Número de galaxias'] = (
    classification_distribution['Código']
    .map(pd.Series(ensemble_predictions).value_counts())
    .fillna(0)
    .astype(int)
)
classification_distribution['Porcentaje'] = (
    100 * classification_distribution['Número de galaxias'] / len(ensemble_predictions)
)
classification_distribution = classification_distribution.sort_values(
    'Número de galaxias', ascending=False
).reset_index(drop=True)

print('Distribución de las galaxias según el ensemble:')
display(classification_distribution[['Código', 'Clase', 'Número de galaxias', 'Porcentaje']])

# Gráfica de la distribución.
plt.figure(figsize=(12, 6))
ax = sns.barplot(
    data=classification_distribution,
    x='Clase',
    y='Número de galaxias'
)
plt.xticks(rotation=45, ha='right')
plt.ylabel('Número de galaxias')
plt.xlabel('Clase predicha')
plt.title('Distribución de las galaxias según enyasantos/galaxy-classification-v02')
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)
plt.tight_layout()
plt.show()

# Confianza del ensemble: probabilidad promedio asignada a la clase predicha.
print(f'Confianza media del ensemble: {ensemble_confidence.mean():.3f}')
print(f'Confianza mediana del ensemble: {np.median(ensemble_confidence):.3f}')

# Guardamos los resultados por muestra para poder relacionarlos después con
# otras variables del HDF5 (por ejemplo, espectros).
df_classification = pd.DataFrame(ensemble_probabilities, columns=[
    f'P_{i}_{GALAXY10_CLASSES[i]}' for i in range(10)
])
df_classification['Clase_predicha'] = ensemble_predictions
df_classification['Clase_predicha_nombre'] = predicted_class_names
df_classification['Confianza'] = ensemble_confidence

df_classification.head()


In [ ]:
# ==========================================
# 5. REDUCCIÓN DE DIMENSIONALIDAD (PCA - 95%)
# ==========================================
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

pca = PCA(n_components=0.95, random_state=42)
features_pca = pca.fit_transform(features_scaled)

print(
    f'Número de componentes retenidas para el 95% de varianza: '
    f'{features_pca.shape[1]}'
)
print(f'Varianza explicada acumulada: {pca.explained_variance_ratio_.sum():.4f}')


In [ ]:
# DataFrame con las componentes principales
pc_columns = [f'PC{i+1}' for i in range(features_pca.shape[1])]
df_pca = pd.DataFrame(features_pca, columns=pc_columns)
print(df_pca.shape)
df_pca.head()


## Etiquetas

El código de carga proporcionado no recupera una variable objetivo de las imágenes. Por ello no se inventan etiquetas. En su lugar, la nueva sección de clasificación usa las **predicciones del propio ensemble** para describir la distribución estimada de tipos de galaxias.


In [ ]:
# ==========================================
# 6. VISUALIZACIÓN DE LOS DESCRIPTORES
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# A. Distribución de las primeras componentes principales
num_pcs_to_plot = min(4, features_pca.shape[1])
df_first_pcs = df_pca.iloc[:, :num_pcs_to_plot]
sns.boxplot(data=df_first_pcs, ax=axes[0])
axes[0].set_title('Distribución de las primeras Componentes Principales')
axes[0].set_xlabel('Componentes Principales')
axes[0].set_ylabel('Valor de la Componente')

# B. Scatterplot PC1 vs PC2
if features_pca.shape[1] >= 2:
    axes[1].scatter(df_pca['PC1'], df_pca['PC2'], s=35, alpha=0.8)
    var_pc1 = pca.explained_variance_ratio_[0] * 100
    var_pc2 = pca.explained_variance_ratio_[1] * 100
    axes[1].set_title('Proyección PCA 2D (PC1 vs PC2)')
    axes[1].set_xlabel(f'PC1 ({var_pc1:.2f}% Varianza Explicada)')
    axes[1].set_ylabel(f'PC2 ({var_pc2:.2f}% Varianza Explicada)')
    axes[1].grid(True, linestyle='--', alpha=0.5)
else:
    axes[1].text(0.5, 0.5, 'PCA produjo una sola componente', ha='center', va='center')
    axes[1].set_axis_off()

plt.tight_layout()
plt.show()


In [ ]:
# # ==========================================
# # 7. RESUMEN DE LA INFORMACIÓN ESPECTRAL
# # ==========================================
# print('Wave:')
# print(f'  shape = {wave.shape}')
# print(f'  rango = [{np.nanmin(wave):.4g}, {np.nanmax(wave):.4g}]')
# print('Flux:')
# print(f'  shape = {flux.shape}')
# print(f'  rango = [{np.nanmin(flux):.4g}, {np.nanmax(flux):.4g}]')
